# Acoustic PDM — Phase 4: Notebook 05
## Multi-Model Comparative Evaluation & SOTA Benchmarking

**Project:** Acoustic Predictive Maintenance (Acoustic PDM)  
**Dataset:** Hitachi MIMII — 4 industrial machine types (**Fan, Pump, Slider, Valve**) at 6 dB SNR, Machine ID 00  
**Input:** Preprocessed tensors from **NB03** + Trained model checkpoints from **NB04**  
**Runtime:** GPU Recommended (Kaggle T4 x 2 or P100)  

---

### What is this notebook about?

In Phase 3 (Notebook 04), we trained our complete model suite:
- **Conv2D-AE** across all 4 machines
- **FC-AE** and **LSTM-AE** on Fan
- **Shallow Baselines**: Isolation Forest, One-Class SVM, XGBoost
- **Dual-Stage Deep Hybrid** (Conv2D-AE + Latent IF) across all 4 machines

In this notebook, we evaluate **all 7 models head-to-head** using industry-standard anomaly detection metrics.

### Evaluation Framework:

1. **Anomaly Scoring** — Per-block anomaly scores across all models
2. **Threshold Calibration** — $\theta = P_{95}$ on held-out normal validation data ($\leq 5\%$ FPR guarantee)
3. **SOTA Metrics** — ROC-AUC, pAUC (max FPR = 10%), Precision, Recall, F1-Score
4. **Cross-Model Comparison Table** — `reports/model_comparison_table.csv`
5. **Multi-Model ROC Curves** — `reports/multi_model_roc_curves.png`
6. **Multi-Machine Generalization** — Conv2D-AE + Hybrid across Fan, Pump, Slider, Valve
7. **Explainable AI (XAI)** — Difference spectrogram heatmaps pinpointing fault frequency bands

---
### Step 0: Environment Bootstrap & Input Discovery

Seeds for reproducibility, GPU detection, and path discovery for both NB03 preprocessed data and NB04 model checkpoints.

> **Kaggle Input:** Attach NB03 output (preprocessed `.npy` tensors) **and** NB04 output (trained model checkpoints `.pth`/`.joblib`) as input datasets.

In [ ]:
import os
import gc
import random
import time
from pathlib import Path
import numpy as np
import torch

# ── Reproducibility Seeds ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ── Path Discovery ──
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    # Find preprocessed data (from NB03 output)
    _data_dir = None
    for _root, _dirs, _ in os.walk("/kaggle/input"):
        if "processed" in _dirs:
            _data_dir = Path(_root) / "processed"
            break
        elif "fan" in _dirs and "pump" in _dirs:
            _data_dir = Path(_root)
            break
    if _data_dir is None:
        _data_dir = Path("/kaggle/working/data/processed")
    PROCESSED_DIR = _data_dir

    # Find model checkpoints (from NB04 output)
    _models_dir = None
    for _root, _dirs, _files in os.walk("/kaggle/input"):
        if any(f.endswith(".pth") for f in _files):
            _models_dir = Path(_root)
            break
    if _models_dir is None:
        _models_dir = Path("/kaggle/working/models")
    MODELS_DIR = _models_dir
else:
    _cwd = Path(os.getcwd()).resolve()
    PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
    MODELS_DIR = PROJECT_ROOT / "models"

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print(f"\U0001f52c ACOUSTIC PDM \u2014 PHASE 4 EVALUATION ENGINE")
print(f"Environment:    {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Device:         {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Processed Data: {PROCESSED_DIR}")
print(f"Checkpoints:    {MODELS_DIR}")
print(f"Reports Output: {REPORTS_DIR}")
print("=" * 70)

---
### Step 1: Import Libraries

Import scientific computing, visualization, and evaluation frameworks.

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support

print("\u2713 Libraries successfully imported.")

---
### Step 2: Model Architectures & Dataset Wrapper (Inline)

Identical architecture definitions to NB04 — required to load saved `.pth` state dictionaries. All models operate on `(Batch, 1, 128, 5)` tensor blocks with 32-dimensional bottleneck $z \in \mathbb{R}^{32}$.

In [ ]:
class AcousticTensorDataset(Dataset):
    def __init__(self, data_array):
        if isinstance(data_array, (str, Path)):
            self.data = np.load(str(data_array)).astype(np.float32)
        else:
            self.data = data_array.astype(np.float32)
        if self.data.ndim == 3:
            self.data = np.expand_dims(self.data, axis=1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx])


class Conv2DAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
        )
        self.encoder_fc = nn.Linear(128 * 16 * 1, latent_dim)
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128 * 16 * 1), nn.LeakyReLU(0.2, inplace=True)
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=(1, 1)),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        return self.encoder_fc(torch.flatten(h, start_dim=1))

    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 128, 16, 1)
        return self.decoder_conv(h)

    def forward(self, x):
        return self.decode(self.encode(x))


class FCAutoencoder(nn.Module):
    def __init__(self, input_dim=640, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, input_dim)
        )

    def encode(self, x):
        return self.encoder(x.view(x.size(0), -1))

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 128, 5)

    def forward(self, x):
        return self.decode(self.encode(x))


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_mels=128, context_frames=5, hidden_dim=64, latent_dim=32):
        super().__init__()
        self.context_frames = context_frames
        self.encoder_lstm = nn.LSTM(input_size=n_mels, hidden_size=hidden_dim, num_layers=2, batch_first=True)
        self.encoder_fc = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=2, batch_first=True)
        self.output_fc = nn.Linear(hidden_dim, n_mels)

    def encode(self, x):
        seq = x.squeeze(1).permute(0, 2, 1)
        _, (h_n, _) = self.encoder_lstm(seq)
        return self.encoder_fc(h_n[-1])

    def decode(self, z):
        h = self.decoder_fc(z).unsqueeze(1).repeat(1, self.context_frames, 1)
        out, _ = self.decoder_lstm(h)
        return self.output_fc(out).permute(0, 2, 1).unsqueeze(1)

    def forward(self, x):
        return self.decode(self.encode(x))


print(f"\u2713 Model architectures defined (Conv2D-AE: {sum(p.numel() for p in Conv2DAutoencoder().parameters()):,} params)")

---
### Step 3: Load Test Data & Model Checkpoints

Load preprocessed test tensors (from NB03) and all trained model checkpoints (from NB04) for evaluation.

In [ ]:
MACHINES = ["fan", "pump", "slider", "valve"]

def load_machine_tensors(machine_name, base_dir=PROCESSED_DIR):
    m_dir = Path(base_dir) / machine_name
    train_normal = np.load(m_dir / "train_normal.npy").astype(np.float32)
    val_normal   = np.load(m_dir / "val_normal.npy").astype(np.float32)
    test_normal  = np.load(m_dir / "test_normal.npy").astype(np.float32)
    test_anomaly = np.load(m_dir / "test_anomaly.npy").astype(np.float32)
    return train_normal, val_normal, test_normal, test_anomaly

# \u2500\u2500 Load Conv2D-AE models (all 4 machines) \u2500\u2500
conv2d_models = {}
for machine in MACHINES:
    m = Conv2DAutoencoder(latent_dim=32)
    m.load_state_dict(torch.load(str(MODELS_DIR / f"best_conv2d_ae_{machine}.pth"), map_location=DEVICE))
    m = m.to(DEVICE).eval()
    conv2d_models[machine] = m
    print(f"  \u2713 Conv2D-AE [{machine.upper()}] loaded")

# \u2500\u2500 Load Fan-specific baseline models \u2500\u2500
fc_model = FCAutoencoder(input_dim=640, latent_dim=32)
fc_model.load_state_dict(torch.load(str(MODELS_DIR / "best_fc_ae_fan.pth"), map_location=DEVICE))
fc_model = fc_model.to(DEVICE).eval()
print("  \u2713 FC-AE [FAN] loaded")

lstm_model = LSTMAutoencoder(n_mels=128, context_frames=5, hidden_dim=64, latent_dim=32)
lstm_model.load_state_dict(torch.load(str(MODELS_DIR / "best_lstm_ae_fan.pth"), map_location=DEVICE))
lstm_model = lstm_model.to(DEVICE).eval()
print("  \u2713 LSTM-AE [FAN] loaded")

# \u2500\u2500 Load Shallow baselines \u2500\u2500
if_shallow = joblib.load(str(MODELS_DIR / "shallow_iforest_fan.joblib"))
ocsvm      = joblib.load(str(MODELS_DIR / "shallow_ocsvm_fan.joblib"))
xgb_clf    = joblib.load(str(MODELS_DIR / "supervised_xgb_fan.joblib"))
print("  \u2713 Shallow baselines (IF, OC-SVM, XGBoost) loaded")

# \u2500\u2500 Load Hybrid Latent IF models (all 4 machines) \u2500\u2500
hybrid_if_models = {}
for machine in MACHINES:
    hybrid_if_models[machine] = joblib.load(str(MODELS_DIR / f"hybrid_latent_if_{machine}.joblib"))
print("  \u2713 Hybrid Latent IF models loaded (all 4 machines)")

print("\n\u2713 All model checkpoints loaded successfully.")

---
### Step 4: Scoring & Metric Utility Functions

Define reusable scoring engines:
- **Reconstruction Error** $S_{\text{recon}}$: Per-block MSE between input and autoencoder output.
- **Latent Outlier Score** $S_{\text{latent}}$: `−score_samples(z)` from the Latent Isolation Forest.
- **Hybrid Fusion** $S_{\text{final}} = 0.6 \cdot \tilde{S}_{\text{recon}} + 0.4 \cdot \tilde{S}_{\text{latent}}$
- **Threshold Calibration** $\theta = P_{95}$ on held-out normal validation data.

In [ ]:
def compute_reconstruction_error(model, data_array, batch_size=256, device=DEVICE):
    """Compute per-block MSE reconstruction error S_recon."""
    model = model.to(device).eval()
    loader = DataLoader(AcousticTensorDataset(data_array), batch_size=batch_size, shuffle=False)
    errors = []
    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            recon = model(x)
            mse = ((x - recon) ** 2).view(x.size(0), -1).mean(dim=1)
            errors.append(mse.cpu().numpy())
    return np.concatenate(errors, axis=0)

def extract_latents(model, data_array, batch_size=256, device=DEVICE):
    """Extract 32-dim latent vectors from encoder bottleneck."""
    model = model.to(device).eval()
    loader = DataLoader(AcousticTensorDataset(data_array), batch_size=batch_size, shuffle=False)
    latents = []
    with torch.no_grad():
        for x in loader:
            z = model.encode(x.to(device))
            latents.append(z.cpu().numpy())
    return np.concatenate(latents, axis=0)

def compute_hybrid_score(s_recon, s_latent, alpha=0.6):
    """Fuse Min-Max normalized reconstruction + latent scores."""
    def min_max(arr):
        return (arr - arr.min()) / (arr.max() - arr.min() + 1e-10)
    return alpha * min_max(s_recon) + (1 - alpha) * min_max(s_latent)

def calibrate_threshold(val_normal_scores, percentile=95.0):
    """Calibrate decision threshold: theta = P_95 on normal validation data."""
    return float(np.percentile(val_normal_scores, percentile))

def compute_metrics(y_true, y_scores, threshold=None):
    """Compute ROC-AUC, pAUC (10% max FPR), and optionally Precision/Recall/F1."""
    auc = roc_auc_score(y_true, y_scores)
    pauc = roc_auc_score(y_true, y_scores, max_fpr=0.1)
    result = {"ROC-AUC": round(auc, 4), "pAUC (10%)": round(pauc, 4)}
    if threshold is not None:
        y_pred = (y_scores >= threshold).astype(int)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        result.update({"Precision": round(prec, 4), "Recall": round(rec, 4), "F1": round(f1, 4)})
    return result

print("\u2713 Scoring & metric utilities defined.")

---
### Step 5: Compute Anomaly Scores — Fan Benchmark (All 7 Models)

Evaluate all 7 models on the **Fan** test set (normal + anomaly blocks). For fair comparison, all models are scored on the same evaluation subset.

Models scored:
1. **Isolation Forest** (Shallow) — `−score_samples(X_flat)`
2. **One-Class SVM** (Shallow) — `−decision_function(X_flat)`
3. **XGBoost** (Supervised) — `predict_proba(X_flat)[:, 1]`
4. **LSTM-AE** — MSE reconstruction error
5. **FC-AE** — MSE reconstruction error
6. **Conv2D-AE** (Standalone) — MSE reconstruction error
7. **Hybrid** (Conv2D-AE + Latent IF) — $0.6 \cdot \tilde{S}_{\text{recon}} + 0.4 \cdot \tilde{S}_{\text{latent}}$

In [ ]:
print("=" * 70)
print("  COMPUTING ANOMALY SCORES \u2014 FAN BENCHMARK (ALL 7 MODELS)")
print("=" * 70)

_, val_fan, test_fan_norm, test_fan_anom = load_machine_tensors("fan")

# Subsample for consistent and fast evaluation across all models
MAX_EVAL = 10000
np.random.seed(SEED)
idx_n = np.random.choice(len(test_fan_norm), min(MAX_EVAL, len(test_fan_norm)), replace=False)
idx_a = np.random.choice(len(test_fan_anom), min(MAX_EVAL, len(test_fan_anom)), replace=False)
eval_norm = test_fan_norm[idx_n]
eval_anom = test_fan_anom[idx_a]
eval_flat_norm = eval_norm.reshape(len(eval_norm), -1)
eval_flat_anom = eval_anom.reshape(len(eval_anom), -1)

n_norm, n_anom = len(eval_norm), len(eval_anom)
y_true = np.array([0] * n_norm + [1] * n_anom)
print(f"Eval set: {n_norm:,} normal + {n_anom:,} anomaly = {n_norm + n_anom:,} blocks")

# Subsample validation for threshold calibration
val_sub = val_fan[:min(MAX_EVAL, len(val_fan))]
val_flat = val_sub.reshape(len(val_sub), -1)

fan_scores = {}  # model_name -> {"test": scores, "val": val_scores}

# 1. Isolation Forest (Shallow)
print("\n1. Isolation Forest (Shallow)...")
t0 = time.time()
fan_scores["Isolation Forest"] = {
    "test": np.concatenate([-if_shallow.score_samples(eval_flat_norm), -if_shallow.score_samples(eval_flat_anom)]),
    "val": -if_shallow.score_samples(val_flat)
}
print(f"   Done in {time.time()-t0:.1f}s")

# 2. One-Class SVM (Shallow)
print("2. One-Class SVM (Shallow)...")
t0 = time.time()
fan_scores["One-Class SVM"] = {
    "test": np.concatenate([-ocsvm.decision_function(eval_flat_norm), -ocsvm.decision_function(eval_flat_anom)]),
    "val": -ocsvm.decision_function(val_flat)
}
print(f"   Done in {time.time()-t0:.1f}s")

# 3. XGBoost (Supervised)
print("3. XGBoost (Supervised)...")
t0 = time.time()
fan_scores["XGBoost (Supervised)"] = {
    "test": np.concatenate([xgb_clf.predict_proba(eval_flat_norm)[:, 1], xgb_clf.predict_proba(eval_flat_anom)[:, 1]]),
    "val": xgb_clf.predict_proba(val_flat)[:, 1]
}
print(f"   Done in {time.time()-t0:.1f}s")

# 4. LSTM-AE
print("4. LSTM-AE...")
t0 = time.time()
fan_scores["LSTM-AE"] = {
    "test": np.concatenate([compute_reconstruction_error(lstm_model, eval_norm), compute_reconstruction_error(lstm_model, eval_anom)]),
    "val": compute_reconstruction_error(lstm_model, val_sub)
}
print(f"   Done in {time.time()-t0:.1f}s")

# 5. FC-AE
print("5. FC-AE...")
t0 = time.time()
fan_scores["FC-AE"] = {
    "test": np.concatenate([compute_reconstruction_error(fc_model, eval_norm), compute_reconstruction_error(fc_model, eval_anom)]),
    "val": compute_reconstruction_error(fc_model, val_sub)
}
print(f"   Done in {time.time()-t0:.1f}s")

# 6. Conv2D-AE (Standalone)
print("6. Conv2D-AE (Standalone)...")
t0 = time.time()
s_recon_norm = compute_reconstruction_error(conv2d_models["fan"], eval_norm)
s_recon_anom = compute_reconstruction_error(conv2d_models["fan"], eval_anom)
s_recon_val  = compute_reconstruction_error(conv2d_models["fan"], val_sub)
fan_scores["Conv2D-AE"] = {
    "test": np.concatenate([s_recon_norm, s_recon_anom]),
    "val": s_recon_val
}
print(f"   Done in {time.time()-t0:.1f}s")

# 7. Hybrid (Conv2D-AE + Latent IF)
print("7. Hybrid (Conv2D-AE + Latent IF)...")
t0 = time.time()
z_norm = extract_latents(conv2d_models["fan"], eval_norm)
z_anom = extract_latents(conv2d_models["fan"], eval_anom)
z_val  = extract_latents(conv2d_models["fan"], val_sub)
s_lat_norm = -hybrid_if_models["fan"].score_samples(z_norm)
s_lat_anom = -hybrid_if_models["fan"].score_samples(z_anom)
s_lat_val  = -hybrid_if_models["fan"].score_samples(z_val)

fan_scores["Hybrid (Conv2D-AE + IF)"] = {
    "test": compute_hybrid_score(np.concatenate([s_recon_norm, s_recon_anom]),
                                  np.concatenate([s_lat_norm, s_lat_anom]), alpha=0.6),
    "val": compute_hybrid_score(s_recon_val, s_lat_val, alpha=0.6)
}
print(f"   Done in {time.time()-t0:.1f}s")

print("\n\u2713 All 7 models scored successfully.")

---
### Step 6: Cross-Model Comparison Table

Compute ROC-AUC, pAUC (10%), Precision, Recall, and F1-Score for all 7 models. Threshold $\theta = P_{95}$ is calibrated per-model on held-out normal validation scores.

In [ ]:
MODEL_ORDER = [
    "Isolation Forest",
    "One-Class SVM",
    "XGBoost (Supervised)",
    "LSTM-AE",
    "FC-AE",
    "Conv2D-AE",
    "Hybrid (Conv2D-AE + IF)",
]

results = []

for name in MODEL_ORDER:
    scores = fan_scores[name]
    theta = calibrate_threshold(scores["val"], percentile=95.0)
    metrics = compute_metrics(y_true, scores["test"], threshold=theta)
    metrics["Model"] = name
    metrics["Threshold"] = round(theta, 6)
    results.append(metrics)

df_results = pd.DataFrame(results)
df_results = df_results[["Model", "ROC-AUC", "pAUC (10%)", "Precision", "Recall", "F1", "Threshold"]]

# Save CSV
csv_path = REPORTS_DIR / "model_comparison_table.csv"
df_results.to_csv(csv_path, index=False)

print("=" * 90)
print("  CROSS-MODEL BENCHMARK COMPARISON TABLE \u2014 FAN (6 dB SNR)")
print("=" * 90)
print(df_results.to_string(index=False))
print("=" * 90)
print(f"\u2713 Saved to: {csv_path.name}")

---
### Step 7: Multi-Model ROC Curve Comparison

Visualize the discrimination power of all 7 models in a single ROC plot. The Dual-Stage Hybrid should dominate the upper-left corner (highest true positive rate at lowest false positive rate).

In [ ]:
MODEL_COLORS = {
    "Isolation Forest": "#d62728",
    "One-Class SVM": "#9467bd",
    "XGBoost (Supervised)": "#8c564b",
    "LSTM-AE": "#2ca02c",
    "FC-AE": "#ff7f0e",
    "Conv2D-AE": "#1f77b4",
    "Hybrid (Conv2D-AE + IF)": "#e7298a",
}

MODEL_STYLES = {
    "Isolation Forest": {"lw": 1.5, "ls": ":"},
    "One-Class SVM": {"lw": 1.5, "ls": ":"},
    "XGBoost (Supervised)": {"lw": 1.5, "ls": "-."},
    "LSTM-AE": {"lw": 2, "ls": "--"},
    "FC-AE": {"lw": 2, "ls": "--"},
    "Conv2D-AE": {"lw": 2.5, "ls": "-"},
    "Hybrid (Conv2D-AE + IF)": {"lw": 3, "ls": "-"},
}

fig, ax = plt.subplots(figsize=(10, 8))

for name in MODEL_ORDER:
    scores = fan_scores[name]["test"]
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc_val = roc_auc_score(y_true, scores)
    style = MODEL_STYLES[name]
    ax.plot(fpr, tpr, color=MODEL_COLORS[name], lw=style["lw"], linestyle=style["ls"],
            label=f"{name} (AUC={auc_val:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label="Random Chance (AUC=0.500)")
ax.set_xlabel("False Positive Rate (FPR)", fontsize=13)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=13)
ax.set_title("Multi-Model ROC Curve Comparison \u2014 Fan (6 dB SNR)", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
roc_path = REPORTS_DIR / "multi_model_roc_curves.png"
plt.savefig(roc_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"\u2713 Saved ROC curves to: {roc_path.name}")

---
### Step 8: Multi-Machine Generalization (Conv2D-AE + Hybrid)

Evaluate the primary **Conv2D-AE** and **Dual-Stage Hybrid** across all 4 industrial machine types (**Fan**, **Pump**, **Slider**, **Valve**) to assess cross-asset generalization.

In [ ]:
print("=" * 70)
print("  MULTI-MACHINE EVALUATION (Conv2D-AE + HYBRID)")
print("=" * 70)

multi_results = []

for machine in MACHINES:
    print(f"\nEvaluating: {machine.upper()}...")
    _, val_m, test_m_norm, test_m_anom = load_machine_tensors(machine)

    # Subsample
    np.random.seed(SEED)
    e_norm = test_m_norm[:min(MAX_EVAL, len(test_m_norm))]
    e_anom = test_m_anom[:min(MAX_EVAL, len(test_m_anom))]
    v_sub  = val_m[:min(MAX_EVAL, len(val_m))]
    y_m = np.array([0] * len(e_norm) + [1] * len(e_anom))

    # Conv2D-AE standalone
    s_r_n = compute_reconstruction_error(conv2d_models[machine], e_norm)
    s_r_a = compute_reconstruction_error(conv2d_models[machine], e_anom)
    s_r_v = compute_reconstruction_error(conv2d_models[machine], v_sub)
    theta_conv = calibrate_threshold(s_r_v)
    m_conv = compute_metrics(y_m, np.concatenate([s_r_n, s_r_a]), theta_conv)

    # Hybrid
    z_n = extract_latents(conv2d_models[machine], e_norm)
    z_a = extract_latents(conv2d_models[machine], e_anom)
    z_v = extract_latents(conv2d_models[machine], v_sub)
    sl_n = -hybrid_if_models[machine].score_samples(z_n)
    sl_a = -hybrid_if_models[machine].score_samples(z_a)
    sl_v = -hybrid_if_models[machine].score_samples(z_v)

    s_hybrid = compute_hybrid_score(np.concatenate([s_r_n, s_r_a]), np.concatenate([sl_n, sl_a]), alpha=0.6)
    s_hybrid_v = compute_hybrid_score(s_r_v, sl_v, alpha=0.6)
    theta_hyb = calibrate_threshold(s_hybrid_v)
    m_hyb = compute_metrics(y_m, s_hybrid, theta_hyb)

    multi_results.append({
        "Machine": machine.upper(),
        "Conv2D-AE AUC": m_conv["ROC-AUC"],
        "Conv2D-AE F1": m_conv["F1"],
        "Hybrid AUC": m_hyb["ROC-AUC"],
        "Hybrid pAUC": m_hyb["pAUC (10%)"],
        "Hybrid F1": m_hyb["F1"],
    })
    print(f"  Conv2D-AE AUC: {m_conv['ROC-AUC']:.4f} | Hybrid AUC: {m_hyb['ROC-AUC']:.4f} | Hybrid F1: {m_hyb['F1']:.4f}")

    # Cleanup
    del val_m, test_m_norm, test_m_anom, e_norm, e_anom, v_sub
    gc.collect()

df_multi = pd.DataFrame(multi_results)
multi_csv = REPORTS_DIR / "multi_machine_hybrid_results.csv"
df_multi.to_csv(multi_csv, index=False)

print("\n" + "=" * 70)
print(df_multi.to_string(index=False))
print("=" * 70)
print(f"\u2713 Saved to: {multi_csv.name}")

---
### Step 9: Explainable AI (XAI) — Difference Spectrogram Heatmaps

Generate `D = |X - \hat{X}|` heatmaps for an anomalous sample from each machine. These pinpoint the **exact frequency bands and temporal instants** driving the anomaly detection decision — critical for maintenance engineers to diagnose the fault type.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 20))

for row, machine in enumerate(MACHINES):
    _, _, _, test_anom = load_machine_tensors(machine)
    ae = conv2d_models[machine]

    # Pick a representative anomalous block
    sample_idx = len(test_anom) // 3  # Deterministic mid-range sample
    sample = test_anom[sample_idx:sample_idx+1]
    sample_t = torch.from_numpy(sample).to(DEVICE)

    ae.eval()
    with torch.no_grad():
        recon_t = ae(sample_t)
    recon = recon_t.cpu().numpy()

    original = sample.squeeze()     # (128, 5)
    reconstructed = recon.squeeze() # (128, 5)
    diff = np.abs(original - reconstructed)

    # Column 1: Original
    ax0 = axes[row, 0]
    im0 = ax0.imshow(original, aspect="auto", origin="lower", cmap="magma")
    ax0.set_title(f"{machine.upper()} \u2014 Original (Anomaly)", fontsize=11, fontweight="bold")
    ax0.set_ylabel("Mel Bin")
    plt.colorbar(im0, ax=ax0, fraction=0.046)

    # Column 2: Reconstructed
    ax1 = axes[row, 1]
    im1 = ax1.imshow(reconstructed, aspect="auto", origin="lower", cmap="magma")
    ax1.set_title(f"{machine.upper()} \u2014 Reconstructed", fontsize=11, fontweight="bold")
    plt.colorbar(im1, ax=ax1, fraction=0.046)

    # Column 3: Difference Heatmap (XAI)
    ax2 = axes[row, 2]
    im2 = ax2.imshow(diff, aspect="auto", origin="lower", cmap="hot")
    ax2.set_title(f"{machine.upper()} \u2014 |X - X\u0302| (XAI)", fontsize=11, fontweight="bold")
    plt.colorbar(im2, ax=ax2, fraction=0.046)

    if row == 3:
        for ax in axes[row]:
            ax.set_xlabel("Time Frame")

    del test_anom
    gc.collect()

plt.tight_layout()
xai_path = REPORTS_DIR / "xai_difference_heatmaps.png"
plt.savefig(xai_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"\u2713 Saved XAI heatmaps to: {xai_path.name}")

---
### Step 10: Phase 4 Summary & Artifact Audit

Audit all generated reports and verify Phase 4 deliverables.

In [ ]:
print("=" * 75)
print("\U0001f3c6 PHASE 4 EVALUATION SUMMARY")
print("=" * 75)

# Best Fan model
best_row = df_results.loc[df_results["ROC-AUC"].idxmax()]
print(f"\n\U0001f947 Best Model (Fan): {best_row['Model']}")
print(f"   ROC-AUC:   {best_row['ROC-AUC']:.4f}")
print(f"   pAUC(10%): {best_row['pAUC (10%)']:.4f}")
print(f"   F1-Score:  {best_row['F1']:.4f}")

# Report artifacts
print(f"\n\U0001f4c4 Generated Reports:")
for f in sorted(REPORTS_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:45s} ({size_kb:.1f} KB)")

print("\n" + "=" * 75)
print("\U0001f389 PHASE 4 COMPLETE: All models benchmarked. Ready for NB06 Interactive Demo!")
print("=" * 75)